# Build 2025 model features

Build resumable scalar features and matched root/all-comment choice sets directly from the normalized 2025 Parquet collection. Production runs require complete collection QA and production NLP; explicitly configured pilot runs are watermarked and must not be used for inference.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

from commentgap_analysis.features import FeatureBuildConfig, build_analysis_features

SEED = int(os.getenv("COMMENTGAP_MODEL_SEED", "20260813"))
DATA_ROOT = Path(os.getenv("COMMENTGAP_DATA_ROOT", "data/scrape_2025"))
OUTPUT_ROOT = Path(os.getenv("COMMENTGAP_FEATURE_ROOT", "model_output/selection_2025/features"))
LOOKBACK = os.getenv("COMMENTGAP_LOOKBACK_ROOT")
INFERENCE_MODE = os.getenv("COMMENTGAP_INFERENCE_MODE", "1") == "1"
ALLOW_INCOMPLETE = os.getenv("COMMENTGAP_ALLOW_INCOMPLETE", "0") == "1"
NLP_MODE = os.getenv("COMMENTGAP_NLP_MODE", "real")
DEVICE = os.getenv("COMMENTGAP_DEVICE", "auto")
SENTIMENT_REVISION = os.getenv("COMMENTGAP_SENTIMENT_REVISION")
EMBEDDING_MODEL_ID = os.getenv("COMMENTGAP_EMBEDDING_MODEL_ID", "BAAI/bge-m3")
EMBEDDING_REVISION = os.getenv("COMMENTGAP_EMBEDDING_REVISION")
EMBEDDING_PROMPT_NAME = os.getenv("COMMENTGAP_EMBEDDING_PROMPT_NAME")
MAX_STORIES = os.getenv("COMMENTGAP_MAX_STORIES")
EXCLUDE_JANUARY = os.getenv("COMMENTGAP_EXCLUDE_JANUARY_WITHOUT_LOOKBACK", "1") == "1"

config = FeatureBuildConfig(
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    year=2025,
    lookback_root=Path(LOOKBACK) if LOOKBACK else None,
    allow_incomplete=ALLOW_INCOMPLETE,
    inference_mode=INFERENCE_MODE,
    nlp_mode=NLP_MODE,
    device=DEVICE,
    sentiment_revision=SENTIMENT_REVISION,
    embedding_model_id=EMBEDDING_MODEL_ID,
    embedding_revision=EMBEDDING_REVISION,
    embedding_max_length=int(os.getenv("COMMENTGAP_EMBEDDING_MAX_LENGTH", "512")),
    embedding_prompt_name=EMBEDDING_PROMPT_NAME,
    sentiment_batch_size=int(os.getenv("COMMENTGAP_SENTIMENT_BATCH", "32")),
    embedding_batch_size=int(os.getenv("COMMENTGAP_EMBEDDING_BATCH", "64")),
    max_stories=int(MAX_STORIES) if MAX_STORIES else None,
    exclude_january_without_lookback=EXCLUDE_JANUARY,
    seed=SEED,
)
config

## Validation and execution plan

1. Validate collection completeness and structural integrity.
2. Compute strictly prior discussion and author-history measures.
3. Checkpoint text, sentiment, article-similarity, and novelty features per story.
4. Residualize CTTR and German SMOG against log length.
5. Create identical curator/audience choice sets with ten deterministic tie draws.

In [ ]:
qa_path = DATA_ROOT / "qa_summary" / "year=2025" / "summary.json"
qa = json.loads(qa_path.read_text())
{
    "passed": qa.get("passed"),
    "allow_incomplete": qa.get("allow_incomplete"),
    "nonterminal_stories": qa.get("nonterminal_stories"),
    "status_counts": qa.get("status_counts"),
    "comment_rows": qa.get("comment_rows"),
    "sticky_descendant_rows": qa.get("sticky_descendant_rows"),
}

In [ ]:
summary = build_analysis_features(config)
summary

## Acceptance checks

The build is complete when both choice-set files, the scalar-feature partitions, tie diagnostics, the shared feature contract, and the provenance manifest exist. Confirm that the watermark is `INFERENCE` before using any result in a paper.

In [ ]:
expected = [
    OUTPUT_ROOT / "choice_set_root.parquet",
    OUTPUT_ROOT / "choice_set_all.parquet",
    OUTPUT_ROOT / "feature_manifest.json",
    OUTPUT_ROOT / "provenance_manifest.json",
    OUTPUT_ROOT / "novelty_validation.json",
]
assert all(path.exists() for path in expected), [str(path) for path in expected if not path.exists()]
json.loads((OUTPUT_ROOT / "provenance_manifest.json").read_text())

## Next step

Knit `06B_stacked_selection_models.Rmd`, then run `06C_xgboost_rankers.ipynb`. For the current incomplete crawl, set `COMMENTGAP_INFERENCE_MODE=0`, `COMMENTGAP_ALLOW_INCOMPLETE=1`, `COMMENTGAP_NLP_MODE=pilot`, and a small `COMMENTGAP_MAX_STORIES`; these outputs remain pilot-only.